<a href="https://colab.research.google.com/github/TheFaridKarami/farid-ai-lab/blob/main/projects/nlp/wnut17-robust-ner/notebooks/wnut17_multi_model_ner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sys
import numpy as np
import torch
from torch import nn
import transformers
from transformers import AutoTokenizer
import datasets

In [ ]:
wnut = datasets.load_dataset("flaitenberger/wnut_17")

In [ ]:
label_list = [label for label in wnut["train"].features["ner_tags"].feature.names]
id2label =   {i:label for i, label in enumerate(label_list)}
label2id =   {label:i for i, label in id2label.items()}


In [113]:
shown = 0
for ex in wnut["train"]:

  if any(t != 0 for t in ex["ner_tags"]):
    print(" ".join(ex["tokens"]))
    print([(tok, id2label[t]) for tok, t in zip(ex["tokens"], ex["ner_tags"])])
    print()
  shown += 1

  if shown == 5:
    break

@paulwalk It 's the view from where I 'm living for two weeks . Empire State Building = ESB . Pretty bad storm here last evening .
[('@paulwalk', 'O'), ('It', 'O'), ("'s", 'O'), ('the', 'O'), ('view', 'O'), ('from', 'O'), ('where', 'O'), ('I', 'O'), ("'m", 'O'), ('living', 'O'), ('for', 'O'), ('two', 'O'), ('weeks', 'O'), ('.', 'O'), ('Empire', 'B-location'), ('State', 'I-location'), ('Building', 'I-location'), ('=', 'O'), ('ESB', 'B-location'), ('.', 'O'), ('Pretty', 'O'), ('bad', 'O'), ('storm', 'O'), ('here', 'O'), ('last', 'O'), ('evening', 'O'), ('.', 'O')]

From Green Newsfeed : AHFA extends deadline for Sage Award to Nov . 5 http://tinyurl.com/24agj38
[('From', 'O'), ('Green', 'O'), ('Newsfeed', 'O'), (':', 'O'), ('AHFA', 'B-group'), ('extends', 'O'), ('deadline', 'O'), ('for', 'O'), ('Sage', 'O'), ('Award', 'O'), ('to', 'O'), ('Nov', 'O'), ('.', 'O'), ('5', 'O'), ('http://tinyurl.com/24agj38', 'O')]

Pxleyes Top 50 Photography Contest Pictures of August 2010 ... http://bit.ly/b

In [ ]:
for split in wnut:
  lens = [len(ex["tokens"]) for ex in wnut[split]]
  print(split, "mean:",round(np.mean(lens),2), "max: ", np.max(lens), "median: ", np.median(lens), "95th pct: ", np.quantile(lens, 0.95))

In [110]:
from collections import Counter

for split in wnut:
  c = Counter(id2label[t] for ex in wnut[split] for t in ex["ner_tags"])
  ents = {l[2:]: n for l , n in c.items() if l.startswith("B-")}
  total = sum(c.values())
  print(split, "tokens:", total, "O share:", round(c["O"] / total, 3))
  print(ents, "total entities:", sum(ents.values()))

train tokens: 62730 O share: 0.95
{'location': 548, 'group': 264, 'corporation': 221, 'person': 660, 'creative-work': 140, 'product': 142} total entities: 1975
validation tokens: 15733 O share: 0.921
{'location': 74, 'creative-work': 105, 'person': 470, 'product': 114, 'group': 39, 'corporation': 34} total entities: 836
test tokens: 23394 O share: 0.926
{'location': 150, 'group': 165, 'person': 429, 'creative-work': 142, 'corporation': 66, 'product': 127} total entities: 1079


In [115]:
tok = AutoTokenizer.from_pretrained("bert-base-cased")

words = ["Pxleyes", "Top", "50", "Photography", "Contest", "#photography"]

enc = tok(words, is_split_into_words= True)

print(tok.convert_ids_to_tokens(enc["input_ids"]))
print(enc.word_ids())

['[CLS]', 'P', '##xley', '##es', 'Top', '50', 'Photography', 'Contest', '#', 'photography', '[SEP]']
[None, 0, 0, 0, 1, 2, 3, 4, 5, 5, None]
